# 14.2 正規化能否簡化？

# 第 14 章：逐項理解現代文字模型的設計

前置：第4章基準。這章像替熟悉的車逐一換零件，每節可獨立進入；不要同時換六項再猜哪項有效。結構／數值可先CPU驗證，品質與成本需匹配預算後訓練。




In [ ]:
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch
from torch import nn
from torch.nn import functional as F

torch.set_num_threads(1)
torch.manual_seed(42)

**先看圖：每張卡片自己調尺度**

沿 feature 軸，不能跨時間偷看別的卡片。

橘色邊框只提示閱讀順序，靜態標註一直保留。系統的減少動態效果設定會停用動畫。


In [ ]:
from IPython.display import SVG, display

display(SVG(filename=str(root / "course/figures/normalization.svg")))

**先想一想：**輸入都為2，LayerNorm與RMSNorm輸出相同嗎？

LayerNorm先搬到平均0再調尺度；RMSNorm只控制平方平均尺度，不減平均值。省一個動作也改了函數，不是完全等價改寫。

**把直覺寫成數學：**RMS(x)=sqrt(mean(x²)+ε)；RMSNorm=x/RMS×gamma。

**最小實驗：**以下程式只處理這節的問題。


In [ ]:
from tiny_perceptron.modern import RMSNorm

x = torch.tensor([[2.0, 2.0, 2.0]])
print("LN", nn.LayerNorm(3)(x), "RMS", RMSNorm(3)(x))

**如何讀結果：**比較不同norm必須重新訓練或明示權重轉換，不能直接拿輸出差說誰更好。

**只改一件事：**只給x加一個常數，看兩者對平移不同。
